## Example 1: async_generator functionality test 1

In [ ]:
import async_generator
from async_generator import async_generator as ag, yield_, yield_from_
import asyncio

# Example 1: Basic async generator creation and usage
print("Example 1: Basic async generator creation and usage")

try:
    print("async_generator module imported successfully")
    print("Provides async generator support for older Python versions")
    
    # Check available components
    attrs = [attr for attr in dir(async_generator) if not attr.startswith('_')]
    print(f"Available components: {attrs}")
    
    # Create a simple async generator
    @ag
    async def simple_async_generator(n):
        for i in range(n):
            await yield_(f"Item {i}")
    
    print("✓ Simple async generator created")
    
    # Test the async generator
    async def test_simple_generator():
        items = []
        async for item in simple_async_generator(3):
            items.append(item)
        return items
    
    # Run the test in Jupyter (using top-level await)
    result = await test_simple_generator()
    print(f"Generated items: {result}")
    assert result == ["Item 0", "Item 1", "Item 2"], result
    
    # Check generator type
    gen = simple_async_generator(2)
    print(f"Generator type: {type(gen).__name__}")
    assert async_generator.isasyncgen(gen) and async_generator.isasyncgenfunction(simple_async_generator)
    assert await gen.__anext__() == "Item 0"
    
    # Clean up
    await gen.aclose()
    
    print("async_generator example 1 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"async_generator example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 1 finished")

## Example 2: async_generator functionality test 2

In [ ]:
from async_generator import async_generator, yield_, yield_from_
import asyncio

# Example 2: Complex async generator with yield_from_
print("Example 2: Complex async generator with yield_from_")

try:
    # Create a base async generator
    @async_generator
    async def number_generator(start, count):
        for i in range(count):
            await asyncio.sleep(0.01)  # Simulate async work
            await yield_(start + i)
    
    # Create a composite async generator using yield_from_
    @async_generator
    async def composite_generator():
        print("Starting first batch...")
        await yield_from_(number_generator(1, 3))
        
        print("Starting second batch...")
        await yield_from_(number_generator(10, 3))
        
        print("Adding final item...")
        await yield_(99)
    
    print("✓ Composite async generator created")
    
    # Test the composite generator
    async def test_composite():
        results = []
        async for value in composite_generator():
            results.append(value)
            print(f"Received: {value}")
        return results
    
    # Run the test
    all_values = await test_composite()
    print(f"All generated values: {all_values}")
    print(f"Total values generated: {len(all_values)}")
    
    # Verify expected pattern
    expected = [1, 2, 3, 10, 11, 12, 99]
    print(f"Values match expected: {all_values == expected}")
    assert all_values == expected, all_values
    
    print("async_generator example 2 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"async_generator example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 2 finished")

## Example 3: async_generator functionality test 3

In [ ]:
from async_generator import async_generator, yield_
import asyncio

# Example 3: Async generator with exception handling and cleanup
print("Example 3: Async generator with exception handling and cleanup")

try:
    events = []  # what the generator did, in order
    
    # Create async generator with resource management
    @async_generator
    async def resource_generator(resource_count):
        resources = []
        try:
            # Setup phase
            for i in range(resource_count):
                resource = f"Resource-{i}"
                resources.append(resource)
                events.append(("acquired", resource))
                print(f"Acquired {resource}")
                await yield_(resource)
                await asyncio.sleep(0.01)  # Simulate processing time
        
        except GeneratorExit:
            events.append(("closed early",))
            print("Generator was closed early")
        except Exception as e:
            print(f"Error in generator: {e}")
        finally:
            # Cleanup phase
            print("Cleaning up resources...")
            for resource in resources:
                events.append(("released", resource))
                print(f"Released {resource}")
    
    print("✓ Resource generator with cleanup created")
    
    # Test normal completion
    async def test_normal_completion():
        print("Testing normal completion:")
        async for resource in resource_generator(3):
            print(f"Using {resource}")
    
    await test_normal_completion()
    assert events == [("acquired", f"Resource-{i}") for i in range(3)] + [("released", f"Resource-{i}") for i in range(3)], events
    events.clear()
    
    # Test early termination
    async def test_early_termination():
        print("\nTesting early termination:")
        gen = resource_generator(5)
        count = 0
        async for resource in gen:
            print(f"Using {resource}")
            count += 1
            if count == 2:  # Stop early
                print("Breaking early...")
                break
        # Leaving the loop does not finalize the generator at a predictable time; aclose() does it now
        await gen.aclose()
    
    await test_early_termination()
    assert events == [
        ("acquired", "Resource-0"), ("acquired", "Resource-1"), ("closed early",),
        ("released", "Resource-0"), ("released", "Resource-1")
    ], events
    
    print("Async generator cleanup and exception handling working")
    print("async_generator example 3 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"async_generator example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 3 finished")

## Example 4: async_generator functionality test 4

In [ ]:
from async_generator import async_generator, yield_
import asyncio

# Example 4: Practical async generator for data processing
print("Example 4: Practical async generator for data processing")

try:
    # Simulate async data source
    class AsyncDataSource:
        def __init__(self, data):
            self.data = data
            self.index = 0
        
        async def fetch_next(self):
            if self.index >= len(self.data):
                return None
            await asyncio.sleep(0.01)  # Simulate network delay
            item = self.data[self.index]
            self.index += 1
            return item
    
    # Create async generator for processing data stream
    @async_generator
    async def process_data_stream(data_source, transform_func):
        processed_count = 0
        while True:
            # Fetch next item from async source
            item = await data_source.fetch_next()
            if item is None:
                break
            
            # Process the item
            try:
                processed_item = transform_func(item)
                processed_count += 1
                await yield_((processed_item, processed_count))
                
            except Exception as e:
                print(f"Error processing item {item}: {e}")
                continue
    
    print("✓ Data processing async generator created")
    
    # Test data processing
    async def test_data_processing():
        # Create sample data source
        sample_data = ["apple", "banana", "cherry", "date", "elderberry"]
        source = AsyncDataSource(sample_data)
        
        # Define transformation function
        def uppercase_transform(item):
            return item.upper()
        
        # Process data stream
        results = []
        async for processed_item, count in process_data_stream(source, uppercase_transform):
            print(f"Processed #{count}: {processed_item}")
            results.append(processed_item)
        
        return results
    
    # Run data processing test
    processed_results = await test_data_processing()
    print(f"Total processed items: {len(processed_results)}")
    print(f"Results: {processed_results}")
    
    # Verify transformation
    expected = ["APPLE", "BANANA", "CHERRY", "DATE", "ELDERBERRY"]
    print(f"Processing successful: {processed_results == expected}")
    assert processed_results == expected, processed_results
    
    print("async_generator example 4 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"async_generator example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 4 finished")

## Example 5: async_generator functionality test 5

In [ ]:
import async_generator
from async_generator import async_generator as ag, yield_, yield_from_, aclosing
from importlib.metadata import version as distribution_version

# Example 5: async_generator package capabilities and compatibility
print("Example 5: async_generator package capabilities and compatibility")

try:
    print("async_generator provides backport of async generators")
    
    # Package analysis
    attrs = [attr for attr in dir(async_generator) if not attr.startswith('_')]
    print(f"Available components: {attrs}")
    
    # Show version information
    print(f"async_generator version: {async_generator.__version__}")
    assert async_generator.__version__ == distribution_version("async_generator"), async_generator.__version__
    assert {'async_generator', 'yield_', 'yield_from_', 'aclosing', 'isasyncgen', 'isasyncgenfunction'} <= set(attrs), attrs
    
    # aclosing() closes a generator when the block ends, even if iteration stopped early
    closed = []
    
    @ag
    async def countdown(n):
        try:
            while n:
                await yield_(n)
                n -= 1
        finally:
            closed.append(True)
    
    async with aclosing(countdown(5)) as numbers:
        async for value in numbers:
            first = value
            break
    assert first == 5 and closed == [True], (first, closed)
    print("✓ aclosing() finalized the generator after an early break")
    
    # Demonstrate compatibility with modern Python
    import sys
    print(f"Python version: {sys.version_info}")
    
    # Show key features
    print("\nKey async_generator features:")
    features = [
        "🔄 Async generator decorator (@async_generator)",
        "⬆️ yield_ function for yielding values", 
        "🔗 yield_from_ for composing generators",
        "🧹 Automatic cleanup and resource management",
        "🐍 Compatibility with older Python versions",
        "🚀 Performance optimized for async workflows",
        "🛡️ Exception handling and early termination support",
        "📦 Drop-in replacement for native async generators"
    ]
    
    for feature in features:
        print(f"  {feature}")
    
    # Compare with native async generators (if available)
    try:
        # Test if native async generators are available
        exec("""
async def native_test():
    yield 1
        """)
        print("\n✓ Native async generators available in this Python version")
        print("async_generator package provides compatibility for older versions")
    except SyntaxError:
        print("\n⚠️ Native async generators not available")
        print("async_generator package enables async generator functionality")
    
    # Usage patterns
    print(f"\nCommon usage patterns:")
    patterns = [
        "Backporting async generators to Python < 3.6",
        "Stream processing with async data sources",
        "Resource management with automatic cleanup",
        "Composing complex async iteration patterns",
        "Building async iterators for web APIs",
        "Processing real-time data streams",
        "Implementing async context managers",
        "Migration path to native async generators"
    ]
    
    for pattern in patterns:
        print(f"• {pattern}")
    
    print(f"\nasync_generator enables async iteration patterns")
    print("async_generator example 5 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"async_generator example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 5 finished")